In [149]:
from pathlib import Path
import sys

%load_ext autoreload
%autoreload 2

dir = Path().resolve().parents[1]

if dir not in sys.path:
    print("directory path is not in the system path")
    sys.path.append(str(dir))
    print("adding directory...")
else:
    print("Directory already exists in the system path")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
directory path is not in the system path
adding directory...


In [150]:
from nn import Unet1D, Returns, RMSELoss
import torch
from torch.utils.data import DataLoader
from scripts import evaluate
import math
import yfinance as yf
from utils import log_transform, posterior_beta, create_dir, inverse_standard
import matplotlib.pyplot as plt
from diffusion import forward, reverse
from arch.univariate import arch_model
import joblib

In [151]:
ticker = "^GSPC"
start_interval = "2016-12-01"
end_interval = "2026-01-01"
interval = "1d"

raw_snp500 = torch.tensor(yf.Ticker(ticker).history(start=start_interval, end=end_interval, interval=interval)["Close"].to_numpy())

split = math.ceil(len(raw_snp500) * 0.15)
val_split = len(raw_snp500) - math.ceil(len(raw_snp500) * 0.15) * 2
test_split = len(raw_snp500) - math.ceil(len(raw_snp500) * 0.15)
train_raw_snp500, val_raw_snp500, test_raw_snp500 = raw_snp500[:val_split], raw_snp500[val_split:test_split], raw_snp500[test_split:]

window_size = 32

train_data = Returns(
  raw_returns=train_raw_snp500,
  window_size=window_size,
  transform=log_transform,
  standard=True
)
val_data = Returns(
  raw_returns=val_raw_snp500,
  window_size=window_size,
  transform=log_transform,
  standard=True
)
test_data = Returns(
  raw_returns=test_raw_snp500,
  window_size=window_size,
  transform=log_transform,
  standard=True
)

train_dataloader = DataLoader(train_data, batch_size=32, shuffle=True, drop_last=True)
val_dataloader = DataLoader(val_data, batch_size=32, shuffle=True, drop_last=True)
test_dataloader = DataLoader(test_data, batch_size=32, shuffle=True, drop_last=True)

In [152]:
encoder_in_channels = [1, 4, 8, 16]
encoder_out_channels = [4, 8, 16, 32]
decoder_in_channels = [32, 16, 8, 4]
decoder_out_channels = [16, 8, 4, 1]
attn_res = 16
n_res_block = 2
T = 1000
num_heads = 4
betas = torch.linspace(1e-4, 2e-2, T)
alpha_hats = torch.cumprod(
  input=1-betas,
  dim=0,
  dtype=torch.float32
)

model = Unet1D(
  attn_res=attn_res,
  n_res_block=n_res_block,
  encoder_in_channels=encoder_in_channels,
  encoder_out_channels=encoder_out_channels,
  decoder_in_channels=decoder_in_channels,
  decoder_out_channels=decoder_out_channels,
  T=T,
  num_heads=num_heads
)
loss_fn = RMSELoss()

In [153]:
SAVE_PATH = dir / "models" / "model_v0.pth"
SAVE_PATH.exists()

True

In [154]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
torch.cuda.manual_seed(42)
device

'cpu'

In [155]:
checkpoint = torch.load(SAVE_PATH, weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

In [156]:
test_result = evaluate(
  data=test_dataloader,
  loss_fn=loss_fn,
  model=model,
  alpha_hats=alpha_hats,
  T=T,
  device=device
)

test_result

0.5761595765749613

In [157]:
test_sample = next(iter(test_dataloader))

In [158]:
# quick sanity check
T = 1000
batch_size = 32

t = torch.randint(0, T, size=(batch_size,))
xt, _ = forward(test_sample, alpha_hats, t)

epsilon_theta = model(xt, t)
epsilon_theta.mean(), epsilon_theta.std()

(tensor(-0.0180, grad_fn=<MeanBackward0>),
 tensor(0.7624, grad_fn=<StdBackward0>))

In [159]:
# reverse process
xT = torch.randn(size=(32, 1, 32))
posterior_betas = torch.tensor([posterior_beta(alpha_hats=alpha_hats, betas=betas, t=t) for t in range(T)])

In [160]:
generated_samples = []
samples = 1
for _ in range(samples):
  sample = reverse(
    xT=xT,
    T=T,
    betas=betas,
    posterior_betas=posterior_betas,
    alpha_bars=alpha_hats,
    model=model
  )
  generated_samples.append(sample)

In [161]:
generated_samples = [sample.squeeze(1) for sample in generated_samples]
generated_samples[0].size()

torch.Size([32, 32])

In [162]:
# generated_samples = torch.cat(tuple(generated_samples), dim=1)
# generated_samples.size()

In [163]:
flatten_generated_samples = generated_samples[0].flatten(0, 1)
flatten_generated_samples.size()

torch.Size([1024])

In [164]:
flatten_generated_samples = flatten_generated_samples.detach().numpy()
type(flatten_generated_samples)

numpy.ndarray

In [165]:
empirical_data = log_transform(train_raw_snp500).detach().numpy()

In [166]:
flatten_generated_samples_inversed = inverse_standard(flatten_generated_samples, empirical_data)

In [167]:
# scaling
flatten_generated_samples_inversed *= 100

### Fit baseline model GARCH

In [168]:
model = arch_model(flatten_generated_samples_inversed, mean="Constant", lags=2, rescale=True)
res = model.fit()
print(res.summary())

Iteration:      1,   Func. Count:      6,   Neg. LLF: 4402.355541079927
Iteration:      2,   Func. Count:     16,   Neg. LLF: 3899.092942467428
Iteration:      3,   Func. Count:     24,   Neg. LLF: 1199.6980785376368
Iteration:      4,   Func. Count:     31,   Neg. LLF: 1199.216308167121
Iteration:      5,   Func. Count:     36,   Neg. LLF: 1199.2209934705495
Iteration:      6,   Func. Count:     42,   Neg. LLF: 1199.2154035765784
Iteration:      7,   Func. Count:     47,   Neg. LLF: 1199.2153795451163
Iteration:      8,   Func. Count:     52,   Neg. LLF: 1199.2153744226293
Iteration:      9,   Func. Count:     56,   Neg. LLF: 1199.2153744216107
Optimization terminated successfully    (Exit mode 0)
            Current function value: 1199.2153744226293
            Iterations: 9
            Function evaluations: 56
            Gradient evaluations: 9
                     Constant Mean - GARCH Model Results                      
Dep. Variable:                      y   R-squared:         

In [169]:
parameters = res.params
parameters

mu          0.020124
omega       0.110396
alpha[1]    0.000000
beta[1]     0.818291
Name: params, dtype: float64

In [170]:
volatility = res._volatility
len(volatility)

1024

In [171]:
PATH = dir / "models" / "garch_diff_v0.pkl"
create_dir(PATH)

joblib.dump(res, PATH)

['D:\\CodingHenry\\research_MBKM\\models\\garch_diff_v0.pkl']

In [172]:
# empirical fitting
empirical_data *= 100

model_empirical = arch_model(empirical_data, mean="Constant", lags=2, rescale=True)
res_empirical = model.fit()
print(res_empirical.summary())

Iteration:      1,   Func. Count:      6,   Neg. LLF: 4402.355541079927
Iteration:      2,   Func. Count:     16,   Neg. LLF: 3899.092942467428
Iteration:      3,   Func. Count:     24,   Neg. LLF: 1199.6980785376368
Iteration:      4,   Func. Count:     31,   Neg. LLF: 1199.216308167121
Iteration:      5,   Func. Count:     36,   Neg. LLF: 1199.2209934705495
Iteration:      6,   Func. Count:     42,   Neg. LLF: 1199.2154035765784
Iteration:      7,   Func. Count:     47,   Neg. LLF: 1199.2153795451163
Iteration:      8,   Func. Count:     52,   Neg. LLF: 1199.2153744226293
Iteration:      9,   Func. Count:     56,   Neg. LLF: 1199.2153744216107
Optimization terminated successfully    (Exit mode 0)
            Current function value: 1199.2153744226293
            Iterations: 9
            Function evaluations: 56
            Gradient evaluations: 9
                     Constant Mean - GARCH Model Results                      
Dep. Variable:                      y   R-squared:         

In [173]:
PATH = dir / "models" / "garch_emp_v0.pkl"
create_dir(PATH)

joblib.dump(res, PATH)

['D:\\CodingHenry\\research_MBKM\\models\\garch_emp_v0.pkl']